# 🛠️ Setup & Data Generation

Welcome to **SQL Practice — Window Functions for Data Engineers**!

This notebook:
1. Verifies your environment (DuckDB, pandas, etc.)
2. Generates all synthetic datasets used by the other notebooks
3. Shows a quick preview of each table

## Datasets
| Table | Rows | Description |
|-------|------|-------------|
| `employees` | 50 | Staff across 6 departments with salaries & hire dates |
| `products` | 20 | Product catalog with categories & prices |
| `sales` | ~250 | Sales-rep performance records (2022–2023) |
| `orders` | 300 | Customer orders linked to products |


## 1 · Environment Check

In [ ]:
import sys
import duckdb
import pandas as pd
import numpy as np

print(f"Python  : {sys.version.split()[0]}")
print(f"DuckDB  : {duckdb.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")


## 2 · Generate Datasets

Run the cell below to create the CSV files in the `data/` folder.

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "../data/generate_data.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


## 3 · Load Data into DuckDB

DuckDB can query CSV files directly via `read_csv_auto()` or register pandas DataFrames as virtual tables.

In [ ]:
import duckdb
import pandas as pd

# Load CSVs
employees = pd.read_csv("../data/employees.csv")
products  = pd.read_csv("../data/products.csv")
sales     = pd.read_csv("../data/sales.csv")
orders    = pd.read_csv("../data/orders.csv")

# Create an in-memory DuckDB connection and register DataFrames
con = duckdb.connect()
con.register("employees", employees)
con.register("products",  products)
con.register("sales",     sales)
con.register("orders",    orders)

print("Tables registered:", con.execute("SHOW TABLES").fetchdf()["name"].tolist())


## 4 · Quick Data Preview

In [ ]:
# Helper to run SQL and display results
def sql(query, n=10):
    return con.execute(query).fetchdf().head(n)


In [ ]:
print("=== employees ===")
display(sql("SELECT * FROM employees LIMIT 5"))

print("\n=== products ===")
display(sql("SELECT * FROM products LIMIT 5"))

print("\n=== sales ===")
display(sql("SELECT * FROM sales LIMIT 5"))

print("\n=== orders ===")
display(sql("SELECT * FROM orders LIMIT 5"))


## 5 · Schema Summary

In [ ]:
for tbl in ["employees", "products", "sales", "orders"]:
    print(f"\n--- {tbl} ---")
    display(con.execute(f"DESCRIBE {tbl}").fetchdf())


## 6 · Your First Window Function

Before jumping into the exercises, here is the canonical syntax for a window function:

```sql
function_name(column)
    OVER (
        PARTITION BY partition_col   -- optional: group rows
        ORDER BY order_col           -- optional: define row order
        ROWS/RANGE BETWEEN ...       -- optional: define frame
    )
```

Let's see one in action — rank employees by salary within their department:

In [ ]:
sql("""
SELECT
    full_name,
    department,
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_salary_rank
FROM employees
ORDER BY department, dept_salary_rank
""")


---
## Next Steps

| Notebook | Topic |
|----------|-------|
| `01_ranking_functions.ipynb` | ROW_NUMBER, RANK, DENSE_RANK, NTILE |
| `02_offset_functions.ipynb` | LAG, LEAD, FIRST_VALUE, LAST_VALUE |
| `03_running_aggregates.ipynb` | SUM/AVG/COUNT OVER, rolling windows |
| `04_advanced_challenges.ipynb` | Real-world data-engineering scenarios |